# Introduction

> - TRIE, also known as a Prefix Tree, is a tree-based data structure used to store and retrieve strings efficiently — especially when dealing with prefix-based queries, like autocomplete or dictionary lookups.
> - Some features,
>   - Each node in a TRIE represents a character.
>   - A path from the root to a node represents a prefix or a complete word.
>   - Common prefixes are shared, making it memory-efficient for large datasets of strings.


Example,

        (root)
          |
          c
          |
          a
        / | \
       t  r  p

> - Applications
>   - Autocomplete systems
>   - Spell checkers
>   - DNA sequence matching
>   - Search engines


# Implementation

In [84]:
class Node:
    def __init__(self):
        self.children = dict()
        self.is_end_of_word = False

- every node will have information about children in form of dictionary, where the key will be some character and value will be a node that the character will point to.

In [85]:
class Trie:
    def __init__(self):
        self.root = Node()
    
    def insert(self, word):
        current_node = self.root

        for char in word:
            if char not in current_node.children:
                current_node.children[char]=Node()
        
            current_node = current_node.children[char]
        current_node.is_end_of_word = True

    def search(self, word):
        current_node = self.root
        for char in word:
            if char not in current_node.children:
                return False
            else:
                current_node=current_node.children[char]
        return current_node.is_end_of_word

    def has_prefix(self, prefix):
        current_node = self.root
        for char in prefix:
            if char not in current_node.children:
                return False
            else:
                current_node=current_node.children[char]
        return True
    
    def delete(self, word):
        self._delete(self.root, word, 0)

    def _delete(self, current_node, word, index):
        if index == len(word):   # means we have traversed entire word
            if not current_node.is_end_of_word:  # if length is complete and still we have not reached end of word
                return False     # then we cant delete anything, return False
            
            # make current node as not the end of word
            current_node.is_end_of_word = False
            return len(current_node.children) == 0  # if true, that means we can delete this node
        
        # get char based on index, and node using char
        char = word[index]
        node = current_node.children.get(char)

        if node is None:
            return False   # if node is None, we dont need to do anything
        
        # recrusively, call this function, with increment in index
        delete_current_node = self._delete(node, word, index+1)

        if delete_current_node:   # this captures True, False and tells whether to delete
            del current_node.children[char]
            return len(current_node.children) == 0 and not current_node.is_end_of_word

        return False

    def starts_with(self, prefix):
        words = []
        current_node = self.root

        for char in prefix:
            if char not in current_node.children:
                return []
            current_node = current_node.children[char]
        # by this line, we are at some node, above which is the prefix

        def _dfs(current_node, path):
            if current_node.is_end_of_word:
                words.append(''.join(path))

            for char, child_node in current_node.children.items():
                _dfs(child_node, path + [char])

        _dfs(current_node, list(prefix))

        return words
    
    def list_words(self):
        pass

# Code Understanding (Recursion)

## Example Visualization: Deleting "CAT" from a Trie

| Call Stack | word / index | Action | Return Value | Trie State (Focus on C-A-T path) |
| :--- | :--- | :--- | :--- | :--- |
| `_delete(ROOT, ..., 0)` | C / 0 | Calls `_delete(C, ..., 1)`. | Returns `False` (Node C is kept). | `ROOT` → **C** |
| `_delete(C, ..., 1)` | A / 1 | Calls `_delete(A, ..., 2)`. | Returns `False` (Node A is kept). | **C** → **A** (A still links to R, V, B) |
| `_delete(A, ..., 2)` | T / 2 | Calls `_delete(T, ..., 3)`. | Returns `True` (Node T is deleted). | **A** → **T** |
| **Base Case** `_delete(T, ..., 3)` | Index `==` Len | **1. Unmark:** `T.is_end_of_word = False`. **2. Check Children:** `len(T.children) = 0`. | **Returns `True`** | Node T is now **unmarked**. |
| **Cleanup** `_delete(A, ..., 2)` | T / 2 | `delete_current_node` is `True`. **1. Delete:** `del A.children['T']`. **2. Check A:** `A.is_end_of_word` is `False`. `len(A.children) = 3` (R, V, B still exist). | **Returns `False`** | T node is **deleted** from A's children. |
| **Cleanup** `_delete(C, ..., 1)` | A / 1 | `delete_current_node` is `False`. | **Returns `False`** | **Deletion stops.** |

## Visualization Table for starts_with("CA")

### Visualization of `_dfs` for `starts_with("CA")`

| Call Stack | `current_node` (Char) | `path` | Action | `words` List |
| :--- | :--- | :--- | :--- | :--- |
| `_dfs(node_A, ['C', 'A'])` | A | `['C', 'A']` | 1. `A.is_end_of_word` is False. 2. Loop through children: 'T', 'R', 'V', 'B'. | `[]` |
| | | | Calls `_dfs(node_T, ['C', 'A', 'T'])` | |
| `_dfs(node_T, ['C', 'A', 'T'])` | T | `['C', 'A', 'T']` | 1. `T.is_end_of_word` is **True**. **Append** "CAT". 2. Loop: No children. | `['CAT']` |
| | | | **Returns** to `node_A` loop. | |
| | | | Calls `_dfs(node_R, ['C', 'A', 'R'])` | |
| `_dfs(node_R, ['C', 'A', 'R'])` | R | `['C', 'A', 'R']` | 1. `R.is_end_of_word` is **True**. **Append** "CAR". 2. Loop: No children. | `['CAT', 'CAR']` |
| | | | **Returns** to `node_A` loop. | |
| | | | Calls `_dfs(node_V, ['C', 'A', 'V', 'E'])` | |
| `_dfs(node_V, ['C', 'A', 'V'])` | V | `['C', 'A', 'V']` | 1. `V.is_end_of_word` is False. 2. Loop: Calls `_dfs(node_E, ['C', 'A', 'V', 'E'])`. | `['CAT', 'CAR']` |
| `_dfs(node_E, ['C', 'A', 'V', 'E'])` | E | `['C', 'A', 'V', 'E']` | 1. `E.is_end_of_word` is **True**. **Append** "CAVE". 2. Loop: No children. | `['CAT', 'CAR', 'CAVE']` |
| | | | **Returns** to `node_V` loop. | |
| | | | **Returns** to `node_A` loop. | |
| | | | Calls `_dfs(node_B, ['C', 'A', 'B'])` | |
| `_dfs(node_B, ['C', 'A', 'B'])` | B | `['C', 'A', 'B']` | 1. `B.is_end_of_word` is **True**. **Append** "CAB". 2. Loop: No children. | `['CAT', 'CAR', 'CAVE', 'CAB']` |
| | | | **Returns** to `node_A` loop. | |
| **Final Return** | A | `['C', 'A']` | Loop finished. **Returns** to `starts_with` function. | `['CAT', 'CAR', 'CAVE', 'CAB']` |

# Visualization

In [86]:
def visualize_trie(root):
    print("(root)")
    stack = [(root, "", "")]  # (node, indent, branch)

    while stack:
        node, indent, branch = stack.pop()

        children = sorted(node.children.items())
        for i in reversed(range(len(children))):  # reversed for correct visual order
            char, child = children[i]
            is_last = i == len(children) - 1
            connector = "└── " if is_last else "├── "
            end_marker = " (end)" if child.is_end_of_word else ""
            print(indent + connector + char + end_marker)

            next_indent = indent + ("    " if is_last else "│   ")
            stack.append((child, next_indent, connector))

In [87]:
trie = Trie()

trie.insert("cap")
trie.insert("car")
trie.insert("cat")

visualize_trie(trie.root)

(root)
└── c
    └── a
        └── t (end)
        ├── r (end)
        ├── p (end)


In [88]:
trie.search("c")

False

In [89]:
trie.has_prefix("c")

True

In [90]:
trie.delete("cat")
visualize_trie(trie.root)

(root)
└── c
    └── a
        └── r (end)
        ├── p (end)


In [91]:
trie.starts_with("ca")

['cap', 'car']